# Assignment 2. 평균이 항상 대표값일까?

**기초생물통계 · 마감 9/21(월) 23:59 · 10점 · 약 20분**

> **질문:** 한 번의 매우 긴 배달(70분)을 포함하면, **평균과 중앙값 중 무엇이 더 크게 변할까?**

### 하는 일은 네 가지입니다 — 코드는 쓰지 않습니다
1. 맨 먼저 **파일 → 드라이브에 사본 저장** (이걸 안 하면 작업이 저장되지 않습니다)
2. 아래 상자를 **위에서 아래로** 차례로 ▶ 실행합니다. 오른쪽 입력 칸이 있는 상자는 칸을 채운 뒤 ▶.
3. **2단계**는 두 번 실행합니다 — 그대로 한 번, `include_long_delay` 체크박스를 켠 뒤 한 번.
4. **4단계**에서 초록색 ✅ 메시지가 나오면 **파일 → 다운로드 → .ipynb 다운로드** 후 LMS에 제출.

코드는 각 상자에 숨겨져 있습니다. 보고 싶으면 상자 오른쪽 위 **"코드 표시"** 를 누르면 되지만, 고치지 않아도 됩니다.


In [ ]:
#@title ✍️ 0단계: 나의 정보와 예측 — 오른쪽 칸을 채운 뒤 ▶ 실행
student_id = ""  #@param {type:"string"}
student_name = ""  #@param {type:"string"}
prediction = "선택하세요"  #@param ["선택하세요", "A. 평균이 더 크게 변한다", "B. 중앙값이 더 크게 변한다", "C. 둘이 비슷하게 변한다"]
prediction_reason = ""  #@param {type:"string"}

problems = []
if not student_id.strip(): problems.append("학번(student_id)을 입력하세요.")
if not student_name.strip(): problems.append("이름(student_name)을 입력하세요.")
if prediction == "선택하세요": problems.append("예측(prediction)을 A/B/C 중에서 고르세요.")
if len(prediction_reason.strip()) < 5: problems.append("예측 이유(prediction_reason)를 한 줄 적어 주세요.")
if problems:
    print("🟨 아직 비어 있는 칸이 있습니다:"); [print("   -", p) for p in problems]
else:
    print(f"✅ {student_id} {student_name} — 예측: {prediction}")
    print(f"   이유: {prediction_reason}")
print("\n다음: 1단계 상자의 ▶를 누르세요. (이 자료는 수업용 가상 배달시간 자료입니다.)")


In [ ]:
#@title ▶ 1단계: 자료 읽기 — 실행만 하세요 (파일 업로드 필요 없음)
from io import StringIO
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

DELIVERY_CSV = r"""order_id,delivery_minutes,case_type
D001,22,usual
D002,24,usual
D003,25,usual
D004,26,usual
D005,27,usual
D006,28,usual
D007,28,usual
D008,29,usual
D009,29,usual
D010,30,usual
D011,30,usual
D012,31,usual
D013,31,usual
D014,32,usual
D015,32,usual
D016,33,usual
D017,33,usual
D018,34,usual
D019,35,usual
D020,37,usual
D021,70,extreme_delay
"""
delivery_all = pd.read_csv(StringIO(DELIVERY_CSV))
n_usual = int((delivery_all["case_type"] == "usual").sum())

print(f"✅ 자료 준비 완료: 배달 {len(delivery_all)}건 = 평소 {n_usual}건 + 70분 지연 1건")
print("배달시간(분)을 작은 값부터:", ", ".join(str(v) for v in sorted(delivery_all["delivery_minutes"].tolist())))
표 = delivery_all.rename(columns={"order_id": "주문번호", "delivery_minutes": "배달시간(분)", "case_type": "구분"}).copy()
표["구분"] = 표["구분"].map({"usual": "평소", "extreme_delay": "70분 지연"})
display(표.tail(6))
print("다음: 2단계 상자를 먼저 그대로 ▶ 실행하세요.")


In [ ]:
#@title ▶ 2단계: 실험 — ① 그대로 ▶ 실행 → ② 오른쪽 체크박스를 켜고 다시 ▶ 실행
include_long_delay = False  #@param {type:"boolean"}

usual = delivery_all[delivery_all["case_type"] == "usual"]["delivery_minutes"]
with_delay = delivery_all["delivery_minutes"]
data = with_delay if include_long_delay else usual

# 두 조건의 요약 — 자동 점검용으로 항상 둘 다 계산해 둔다
ref = {
    "제외": {"평균": usual.mean(), "중앙값": usual.median(), "표준편차": usual.std(ddof=1)},
    "포함": {"평균": with_delay.mean(), "중앙값": with_delay.median(), "표준편차": with_delay.std(ddof=1)},
}
요약표 = pd.DataFrame({
    "조건": ["70분 사례 제외 (평소 20건)", "70분 사례 포함 (21건)"],
    "건수": [len(usual), len(with_delay)],
    "평균": [ref["제외"]["평균"], ref["포함"]["평균"]],
    "중앙값": [ref["제외"]["중앙값"], ref["포함"]["중앙값"]],
    "표준편차": [ref["제외"]["표준편차"], ref["포함"]["표준편차"]],
}).round(2)
display(요약표)

mean_value, median_value = data.mean(), data.median()
cond = "포함" if include_long_delay else "제외"
print(f"지금 그래프의 조건: 70분 사례 {cond}  →  평균 {mean_value:.2f}분, 중앙값 {median_value:.1f}분  (차이 {mean_value - median_value:.2f}분)")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={"width_ratios": [2.2, 1]})
axes[0].hist(data, bins=np.arange(10, 81, 5), color="#4C78A8", edgecolor="white")
axes[0].axvline(mean_value, color="#E45756", linewidth=2.5, label=f"Mean = {mean_value:.1f}")
axes[0].axvline(median_value, color="#54A24B", linewidth=2.5, linestyle="--", label=f"Median = {median_value:.1f}")
axes[0].set_xlabel("Delivery time (minutes)"); axes[0].set_ylabel("Count")
axes[0].set_title(f"Histogram (long delay {'included' if include_long_delay else 'excluded'})")
axes[0].legend(); axes[0].grid(axis="x", alpha=0.2)
axes[1].boxplot(data, vert=True, widths=0.5, patch_artist=True,
                boxprops={"facecolor": "#DCE9F5"}, medianprops={"color": "#54A24B", "linewidth": 2.5},
                flierprops={"marker": "o", "markerfacecolor": "#E45756", "markersize": 8})
axes[1].axhline(mean_value, color="#E45756", linewidth=1.5, linestyle=":", label="Mean")
axes[1].set_ylim(10, 80); axes[1].set_xticks([]); axes[1].set_ylabel("Delivery time (minutes)")
axes[1].set_title("Box plot (dot = outlier)"); axes[1].legend(loc="upper left")
plt.tight_layout(); plt.show()

if not include_long_delay:
    print("👉 이제 오른쪽의 ☐ include_long_delay 체크박스를 켠 뒤, 이 상자의 ▶를 다시 누르세요.")
else:
    print("✅ 두 조건이 모두 표에 있습니다. 위 표의 숫자를 보고 3단계 입력 칸을 채우세요. (그래프: 빨간 실선 = 평균, 초록 점선 = 중앙값, 상자그림의 빨간 점 = 이상값)")


In [ ]:
#@title ✍️ 3단계: 답안 입력 — 2단계 표의 숫자를 옮겨 적고(소수 둘째 자리까지) 문장을 고른 뒤 ▶ 실행
#@markdown **관찰표** — 70분 사례 제외(평소 20건)
mean_without = 0  #@param {type:"number"}
median_without = 0  #@param {type:"number"}
sd_without = 0  #@param {type:"number"}
#@markdown **관찰표** — 70분 사례 포함(21건)
mean_with = 0  #@param {type:"number"}
median_with = 0  #@param {type:"number"}
sd_with = 0  #@param {type:"number"}
#@markdown **문장 완성**
changed_more = "선택하세요"  #@param ["선택하세요", "평균", "중앙값", "둘이 비슷하게"]
better_summary = "선택하세요"  #@param ["선택하세요", "평균", "중앙값"]
reason_sentence = ""  #@param {type:"string"}
prediction_matched = "선택하세요"  #@param ["선택하세요", "같았다", "달랐다"]

ok, notes = True, []
if "ref" not in globals():
    ok = False
    print("🟨 먼저 2단계 상자를 실행하세요 (체크박스를 켠 상태로). 실행한 뒤 이 상자의 ▶를 다시 누르세요.")
else:
    def check(label, typed, true_value):
        global ok
        if typed == 0:
            ok = False; notes.append(f"⚠️ {label}: 입력하지 않았습니다 (0). 2단계 표에서 옮겨 적으세요.")
        elif abs(typed - true_value) > 0.06:
            ok = False; notes.append(f"⚠️ {label}: 입력 {typed} — 2단계 표의 값과 다릅니다. 소수 둘째 자리까지 다시 확인하세요.")
        else:
            notes.append(f"✅ {label}: {typed}")
    check("평균(제외)", mean_without, ref["제외"]["평균"]);   check("중앙값(제외)", median_without, ref["제외"]["중앙값"]);   check("표준편차(제외)", sd_without, ref["제외"]["표준편차"])
    check("평균(포함)", mean_with, ref["포함"]["평균"]);      check("중앙값(포함)", median_with, ref["포함"]["중앙값"]);      check("표준편차(포함)", sd_with, ref["포함"]["표준편차"])
    for s_ in ("changed_more", "better_summary", "prediction_matched"):
        if globals()[s_] == "선택하세요": ok = False; notes.append(f"⚠️ {s_}: 선택하지 않았습니다.")
    if len(reason_sentence.strip()) < 10:
        ok = False; notes.append("⚠️ reason_sentence: 이유를 한 문장(10자 이상)으로 적어 주세요. 힌트 단어: 이상값, 크기를 다 더한다, 순서만 본다")
    print("\n".join(notes))

if ok:
    josa = {"평균": "평균이", "중앙값": "중앙값이", "둘이 비슷하게": "둘이 비슷하게"}[changed_more]
    print("\n📝 완성된 답안")
    print(f"① 70분 지연 사례를 추가하자 {josa} 더 크게 변했다. (평균 {mean_without}분 → {mean_with}분, 중앙값 {median_without}분 → {median_with}분, 표준편차 {sd_without}분 → {sd_with}분)")
    print(f"② 따라서 이 자료에서 평소 배달시간은 {better_summary}으로 설명하는 편이 더 적절하다. 이유: {reason_sentence}")
    print(f"③ 나의 예상({prediction})은 결과와 {prediction_matched}.")
    print("\n다음: 4단계 상자로 가세요.")
else:
    print("\n🟨 ⚠️ 표시된 칸을 고친 뒤 이 상자의 ▶를 다시 누르세요.")


In [ ]:
#@title ✅ 4단계: 제출 점검 — AI·타인 도움 여부를 표시하고 ▶ 실행 → 초록색 메시지가 나오면 다운로드
ai_help = "사용하지 않음"  #@param ["사용하지 않음", "사용함"]
ai_help_detail = ""  #@param {type:"string"}
#@markdown 사용했다면 어디에, 무엇을 확인했는지 한 줄 (예: "평균·중앙값 차이를 ChatGPT에 물어봤고, 표의 숫자로 직접 확인"). 기록 자체가 점수에 포함됩니다.

issues = []
try:
    if not include_long_delay: issues.append("2단계 체크박스 include_long_delay를 켜고 2단계를 다시 실행하세요.")
except NameError:
    issues.append("2단계를 실행하지 않았습니다.")
try:
    if not ok: issues.append("3단계에 ⚠️ 표시가 남아 있습니다. 고친 뒤 3단계를 다시 실행하세요.")
except NameError:
    issues.append("3단계를 실행하지 않았습니다.")
try:
    sid = student_id.strip()
    if not (sid.isdigit() and 6 <= len(sid) <= 10): issues.append("0단계 학번은 숫자만 6~10자리로 입력하세요.")
except NameError:
    issues.append("0단계를 실행하지 않았습니다."); sid = "학번"
if ai_help == "사용함" and len(ai_help_detail.strip()) < 5:
    issues.append("AI·타인 도움을 사용했다면 어디에 어떻게 썼는지 한 줄 적어 주세요.")

if issues:
    print("🟨 제출 전에 다음을 고치세요:"); [print("   -", i) for i in issues]
else:
    print("✅ 제출 준비 완료!")
    print(f"   도움 사용: {ai_help}" + (f" — {ai_help_detail}" if ai_help == "사용함" else ""))
    print("\n제출 절차 (3단계)")
    print("  1) 메뉴 런타임 → '세션 다시 시작 및 모두 실행' → 위에서 아래까지 ⚠️ 없이 도는지 확인 (0~4단계 입력값은 유지됩니다)")
    print("  2) 메뉴 파일 → 다운로드 → '.ipynb 다운로드'  (출력이 보이는 상태로! '모든 출력 지우기'는 누르지 마세요)")
    print(f"  3) 내려받은 파일 이름을  {sid}_A2.ipynb  로 바꾸어 LMS 과제 ② 제출함에 올리기 (마감 9/21 월 23:59)")


---
## 채점 기준 (10점)

| 항목 | 점수 | 기준 |
|---|---:|---|
| 실행 완결성 | 3 | 0~4단계가 순서대로 실행되고 표·그래프 출력이 남아 있음 |
| 지정 변경값 | 2 | `include_long_delay`를 켠 상태로 2단계를 다시 실행함 |
| 관찰과 통계 설명 | 3 | 3단계 답안: 표의 숫자가 맞고, 이유 문장에 개념(이상값·평균의 민감성)이 들어 있음 |
| 한계·재현성·도움 공개 | 2 | 예측과 결과 비교, '모두 실행' 확인, 도움 사용 여부 기록 |

## 문제가 생겼을 때

1. 오류가 난 상자 **바로 위 상자부터** 다시 ▶ 실행한다.
2. 실수로 코드를 건드렸다면 **파일 → 노트 업로드**로 배부본을 다시 올려 **새 사본**에서 시작한다.
3. 세션이 끊겼다는 메시지가 나오면 **런타임 → 세션 다시 시작 및 모두 실행**.
4. 그래도 안 되면 오류 화면을 **지우지 말고 캡처**해서 게시판이나 조교 이메일로 질문한다.

*이 과제의 배달시간 자료는 수업용으로 만든 가상 자료이며 실제 주문·업체·학생과 무관합니다.*
